# Rondas Hídricas

> **Contexto de dominio:** [`docs/fuentes/rondas_hidricas.md`](../../docs/fuentes/rondas_hidricas.md)  
> **Bloque:** A | **Línea:** `rondas_hidricas`  
> **Variable principal:** `caudal` (m³/s)  
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/rondas_hidricas.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "rondas_hidricas"
VARIABLE = "caudal"
UNIDAD = "m³/s"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `rondas_hidricas` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "rondas_hidricas.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/rondas_hidricas.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/rondas_hidricas.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "caudal": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `rondas_hidricas` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_rondas_hidricas.html",
       title="EDA — Rondas Hídricas", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "caudal", title="Rondas Hídricas — caudal (m³/s)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "caudal", period="month")
plt.show()

<!-- ENRICHMENT: rondas_hidricas -->

## 3c. HAND y analisis de frecuencia — Delimitacion de ronda hidrica

El **HAND (Height Above Nearest Drainage)** es un indice geomorfometrico derivado del DEM que expresa la diferencia de elevacion entre cada celda y el drenaje mas cercano. Es la herramienta central para el acotamiento funcional de rondas hidricas (Res. 957/2018):

```
HAND(x,y) = elevacion(x,y) - elevacion(drenaje_mas_cercano)
HAND = 0    : cauce permanente
HAND < 1m   : zona inundacion banca llena (ordinaria)
HAND < Tr100: zona de flujo preferente (inundacion 100 anos)
```

Herramientas: **WhiteboxTools** (algoritmo D8 para red drenaje + HAND) o TauDEM.

**Analisis de frecuencia de caudales maximos (Gumbel):**
```
F(Q) = exp(-exp(-y))    y = (Q - alpha) / beta
Caudal_Tr = alpha - beta * ln(-ln(1 - 1/Tr))
```

In [ ]:
# HAND conceptual + analisis frecuencia Gumbel para caudales maximos
from scipy import stats
np.random.seed(9)

# -- Simulacion HAND transecto de rio (perfil transversal) ----------------
distancia_m = np.linspace(-200, 200, 400)   # m desde el cauce
# Perfil topografico sintetico (valle en V con llanura)
hand_perfil = np.where(
    np.abs(distancia_m) < 30,
    np.abs(distancia_m) * 0.05,             # cauce y zona banca llena
    0.05 * 30 + (np.abs(distancia_m) - 30) * 0.04  # ladera
) + np.random.normal(0, 0.1, 400)
hand_perfil = np.clip(hand_perfil, 0, None)

# Umbrales HAND para ronda hidrica
HAND_BANCA_LLENA = 0.5  # m — inundacion ordinaria
HAND_TR15 = 1.8         # m — periodo retorno 15 anos
HAND_TR100 = 4.2        # m — periodo retorno 100 anos (zona flujo preferente)

# -- Analisis frecuencia Gumbel sobre caudales maximos anuales ------------
n_anos = 35
q_max_anual = np.random.gumbel(loc=80, scale=25, size=n_anos).clip(20)  # m3/s

# Ajuste Gumbel (metodo momentos)
alpha_g = q_max_anual.std() * np.sqrt(6) / np.pi
beta_g  = q_max_anual.mean() - 0.5772 * alpha_g

Tr_periodos = [2, 5, 10, 25, 50, 100]
q_tr = [beta_g - alpha_g * np.log(-np.log(1 - 1/tr)) for tr in Tr_periodos]
df_gumbel = pd.DataFrame({'Tr_anos': Tr_periodos, 'Q_m3s': [round(q,2) for q in q_tr]})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Perfil HAND transecto
ax = axes[0]
ax.fill_between(distancia_m, hand_perfil, alpha=0.3, color='#8B4513')
ax.plot(distancia_m, hand_perfil, color='#8B4513', lw=1)
ax.axhline(HAND_BANCA_LLENA, color='#3498db', ls='-', lw=2, label=f'Banca llena (HAND={HAND_BANCA_LLENA}m)')
ax.axhline(HAND_TR15, color='orange', ls='--', lw=1.5, label=f'Tr=15 anos (HAND={HAND_TR15}m)')
ax.axhline(HAND_TR100, color='red', ls='--', lw=1.5, label=f'Tr=100 anos/ronda hidrica (HAND={HAND_TR100}m)')
ax.set_xlabel('Distancia al cauce (m)'); ax.set_ylabel('HAND (m)')
ax.set_title('HAND — Perfil transversal ronda hidrica\n(WhiteboxTools D8, Res. 957/2018)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel B: Curva de frecuencia Gumbel
ax = axes[1]
Tr_cont = np.linspace(1.1, 200, 500)
q_cont = beta_g - alpha_g * np.log(-np.log(1 - 1/Tr_cont))
ax.plot(Tr_cont, q_cont, lw=2, color='#2980b9', label='Ajuste Gumbel')
ax.scatter([r['Tr_anos'] for _, r in df_gumbel.iterrows()],
           [r['Q_m3s'] for _, r in df_gumbel.iterrows()],
           color='red', zorder=5, s=60, label='Tr de diseno')
ax.axvline(100, color='red', ls='--', lw=1.5, label='Tr=100 anos (ronda hidrica)')
ax.axvline(15, color='orange', ls='--', lw=1, label='Tr=15 anos (Res. 957/2018)')
ax.set_xscale('log')
ax.set_xlabel('Periodo de retorno Tr (anos, log)'); ax.set_ylabel('Caudal maximo (m3/s)')
ax.set_title('Analisis frecuencia Gumbel — Caudales maximos\nRonda hidrica: zona de flujo preferente Tr=100', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('Caudales de diseno por periodo de retorno:')
print(df_gumbel.to_string(index=False))

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `rondas_hidricas` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Rondas Hídricas) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["caudal"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `caudal` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["caudal"], variable="caudal")
if rep.empty:
    print("Sin normas colombianas registradas para 'caudal'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["caudal"], method="linear")
print(f"Faltantes antes: {df["caudal"].isna().sum()} | "
      f"después: {df_clean["caudal"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["caudal"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: rondas_hidricas -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Hidrología / hidráulica

- **Distribuciones extremas:** Gumbel / Pearson III sobre caudales máximos anuales — asumir T=2,5,10,25,50,100 años.
- **Lag ENSO = 3 meses.**
- **Validación con LiDAR / topografía** cuando esté disponible — la ronda hídrica es geomorfológica, no sólo estadística.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Rondas Hídricas (`rondas_hidricas`)
- **Variable analizada:** `caudal` (m³/s)
- **Modelos ejecutados:** SARIMA, PROPHET, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/rondas_hidricas.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.